# Spark + Delta Lake Local Analysis

Simple examples for working with Spark and Delta Lake tables.

In [20]:
from pyspark.sql import SparkSession

# Connect to Spark
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
print("Connected to Spark!")

Connected to Spark!


In [21]:
# Show available databases
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|     demo|
+---------+



In [22]:
# Create a database
spark.sql("CREATE DATABASE IF NOT EXISTS demo").show()

++
||
++
++



In [23]:
# Create a Delta Lake table
spark.sql("""
    CREATE TABLE IF NOT EXISTS demo.sales (
        id INT,
        product STRING,
        amount DOUBLE,
        sale_date DATE
    ) USING DELTA
""").show()

++
||
++
++



In [24]:
# Insert data - use DATE() function for date columns
spark.sql("""
    INSERT INTO demo.sales VALUES 
    (1, 'Widget A', 99.99, DATE('2026-02-22')),
    (2, 'Widget B', 149.99, DATE('2026-02-22')),
    (3, 'Widget C', 199.99, DATE('2026-02-22'))
""").show()

++
||
++
++



In [25]:
# Query the table
spark.sql("SELECT * FROM demo.sales").show()

+---+--------+------------------+----------+
| id| product|            amount| sale_date|
+---+--------+------------------+----------+
|  3|Widget C|            249.99|2026-02-23|
|  3|Widget C|            249.99|2026-02-23|
|  3|Widget C|            249.99|2026-02-23|
|  3|Widget C|            249.99|2026-02-23|
|  4|Widget D|218.69271000000003|2026-02-23|
|  6|Widget F|            499.99|2026-02-23|
|  5|Widget E|323.99190000000004|2026-02-23|
|  5|Widget E|359.99100000000004|2026-02-23|
|  4|Widget D|269.99100000000004|2026-02-23|
|  4|Widget D|218.69271000000003|2026-02-23|
|  2|Widget B|109.34271000000001|2026-02-22|
|  4|Widget D|242.99190000000004|2026-02-23|
|  2|Widget B|109.34271000000001|2026-02-22|
|  5|Widget E|291.59271000000007|2026-02-23|
|  2|Widget B|121.49190000000002|2026-02-22|
|  5|Widget E|291.59271000000007|2026-02-23|
|  2|Widget B|            149.99|2026-02-22|
|  2|Widget B|           134.991|2026-02-22|
|  1|Widget A|             99.99|2026-02-22|
|  3|Widge

In [26]:
# Use DataFrame API
df = spark.table("demo.sales")
df.filter(df.amount > 100).show()

+---+--------+------------------+----------+
| id| product|            amount| sale_date|
+---+--------+------------------+----------+
|  3|Widget C|            249.99|2026-02-23|
|  3|Widget C|            249.99|2026-02-23|
|  3|Widget C|            249.99|2026-02-23|
|  3|Widget C|            249.99|2026-02-23|
|  4|Widget D|218.69271000000003|2026-02-23|
|  6|Widget F|            499.99|2026-02-23|
|  5|Widget E|323.99190000000004|2026-02-23|
|  5|Widget E|359.99100000000004|2026-02-23|
|  4|Widget D|269.99100000000004|2026-02-23|
|  4|Widget D|218.69271000000003|2026-02-23|
|  2|Widget B|109.34271000000001|2026-02-22|
|  4|Widget D|242.99190000000004|2026-02-23|
|  2|Widget B|109.34271000000001|2026-02-22|
|  5|Widget E|291.59271000000007|2026-02-23|
|  2|Widget B|121.49190000000002|2026-02-22|
|  5|Widget E|291.59271000000007|2026-02-23|
|  2|Widget B|            149.99|2026-02-22|
|  2|Widget B|           134.991|2026-02-22|
|  3|Widget C|            199.99|2026-02-22|
+---+-----

In [27]:
# Delta Lake - view table history
spark.sql("DESCRIBE HISTORY demo.sales").show(truncate=False)

+-------+-----------------------+------+--------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------

In [28]:
# Delta Lake - view table details
spark.sql("DESCRIBE DETAIL demo.sales").show(truncate=False)

+------+------------------------------------+------------------------+-----------+---------------------------------------+-----------------------+-----------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+------------------------+
|format|id                                  |name                    |description|location                               |createdAt              |lastModified           |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
+------+------------------------------------+------------------------+-----------+---------------------------------------+-----------------------+-----------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+------------------------+
|delta |9ccdb127-be55-40fd-9442-102b888b5b92|spark_catalog.demo.sales|NULL       |file:/lakehouse/warehouse/d

In [29]:
# Add more data to create a new version
spark.sql("""
    INSERT INTO demo.sales VALUES 
    (4, 'Widget D', 299.99, DATE('2026-02-23')),
    (5, 'Widget E', 399.99, DATE('2026-02-23'))
""").show()
print("New version created!")

++
||
++
++

New version created!


In [30]:
# Time travel - query previous version
# Delta Lake versions start at 0. After the first insert, we should have version 0.
# After adding more data, we have version 1. Let's query version 0.
try:
    print("Querying version 0 (first insert):")
    spark.sql("SELECT * FROM demo.sales VERSION AS OF 0").show()
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you've run the cells above to create data first!")

Querying version 0 (first insert):
+---+-------+------+---------+
| id|product|amount|sale_date|
+---+-------+------+---------+
+---+-------+------+---------+



In [31]:
# Update data (ACID transaction)
spark.sql("""
    UPDATE demo.sales 
    SET amount = amount * 0.9 
    WHERE product LIKE 'Widget%'
""").show()
print("Applied 10% discount to all widgets")

+-----------------+
|num_affected_rows|
+-----------------+
|               22|
+-----------------+

Applied 10% discount to all widgets


In [32]:
# Delete data (ACID transaction)
spark.sql("""
    DELETE FROM demo.sales 
    WHERE amount < 100
""").show()
print("Deleted low-value sales")

+-----------------+
|num_affected_rows|
+-----------------+
|                3|
+-----------------+

Deleted low-value sales


In [33]:
# Merge operation (upsert)
# Create a temporary table with updates
spark.sql("""
    CREATE TEMPORARY VIEW updates AS 
    SELECT 3 AS id, 'Widget C' AS product, 249.99 AS amount, DATE('2026-02-23') AS sale_date
    UNION ALL
    SELECT 6 AS id, 'Widget F' AS product, 499.99 AS amount, DATE('2026-02-23') AS sale_date
""")

# Merge updates into main table
spark.sql("""
    MERGE INTO demo.sales AS target
    USING updates AS source
    ON target.id = source.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""").show()
print("Merged updates: updated existing record and inserted new one")

AnalysisException: [TEMP_TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create the temporary view `updates` because it already exists.
Choose a different name, drop or replace the existing view,  or add the IF NOT EXISTS clause to tolerate pre-existing views. SQLSTATE: 42P07

JVM stacktrace:
org.apache.spark.sql.catalyst.analysis.TempTableAlreadyExistsException
	at org.apache.spark.sql.catalyst.catalog.SessionCatalog.createTempView(SessionCatalog.scala:654)
	at org.apache.spark.sql.execution.command.CreateViewCommand.run(views.scala:145)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResult$lzycompute(commands.scala:79)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResult(commands.scala:77)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.executeCollect(commands.scala:88)
	at org.apache.spark.sql.execution.QueryExecution.$anonfun$eagerlyExecuteCommands$2(QueryExecution.scala:155)
	at org.apache.spark.sql.execution.SQLExecution$.$

In [ ]:
# View all versions in history
spark.sql("DESCRIBE HISTORY demo.sales").show(truncate=False)

+-------+-----------------------+------+--------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------

In [ ]:
# Show final table state
spark.sql("SELECT * FROM demo.sales ORDER BY id").show()

+---+--------+------------------+----------+
| id| product|            amount| sale_date|
+---+--------+------------------+----------+
|  2|Widget B|           134.991|2026-02-22|
|  2|Widget B|121.49190000000002|2026-02-22|
|  2|Widget B|109.34271000000001|2026-02-22|
|  2|Widget B|109.34271000000001|2026-02-22|
|  3|Widget C|            249.99|2026-02-23|
|  3|Widget C|            249.99|2026-02-23|
|  3|Widget C|            249.99|2026-02-23|
|  3|Widget C|            249.99|2026-02-23|
|  4|Widget D|242.99190000000004|2026-02-23|
|  4|Widget D|269.99100000000004|2026-02-23|
|  4|Widget D|218.69271000000003|2026-02-23|
|  4|Widget D|218.69271000000003|2026-02-23|
|  5|Widget E|291.59271000000007|2026-02-23|
|  5|Widget E|291.59271000000007|2026-02-23|
|  5|Widget E|323.99190000000004|2026-02-23|
|  5|Widget E|359.99100000000004|2026-02-23|
|  6|Widget F|            499.99|2026-02-23|
+---+--------+------------------+----------+



In [ ]:
# Vacuum old files (remove old versions)
# For testing/demo purposes, we'll disable the retention check
# WARNING: In production, use at least 168 hours (7 days) retention!

# Disable retention check for this demo
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# Dry run to see what would be deleted (safe - doesn't actually delete)
print("DRY RUN - files that would be deleted:")
spark.sql("VACUUM demo.sales RETAIN 0 HOURS DRY RUN").show(truncate=False)

# Production-safe example (commented out):
# spark.sql("VACUUM demo.sales RETAIN 168 HOURS").show()

DRY RUN - files that would be deleted:
+-----------------------------------------------------------------------------------------------------------+
|path                                                                                                       |
+-----------------------------------------------------------------------------------------------------------+
|file:/lakehouse/warehouse/demo.db/sales/part-00000-27a3d31b-85d5-4750-b091-8e7d698a0688-c000.snappy.parquet|
|file:/lakehouse/warehouse/demo.db/sales/part-00000-9b91605e-e828-44e1-9a6e-3568f9832fa4-c000.snappy.parquet|
|file:/lakehouse/warehouse/demo.db/sales/part-00000-bc880cee-84b2-4e6f-99dc-2c6884f2403a-c000.snappy.parquet|
|file:/lakehouse/warehouse/demo.db/sales/part-00001-72a2f12e-b026-4e51-af28-b468c95e93ae-c000.snappy.parquet|
|file:/lakehouse/warehouse/demo.db/sales/part-00006-6c260b41-bc38-4a48-9d89-29a3f75f2c59-c000.snappy.parquet|
|file:/lakehouse/warehouse/demo.db/sales/part-00000-26005a3b-1829-4953-95da-2782a

In [ ]:
# Show all tables
spark.sql("SHOW TABLES IN demo").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     demo|    sales|      false|
|         |  updates|       true|
+---------+---------+-----------+

